In [1]:
pip install python-docx openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import openpyxl
from docx import Document

# Fayl yollarını lazım gəldikcə dəyişin
WORD_FILE = r"C:\Users\AGS_ASC\Desktop\coordinat\Processing\Bag monitorinq 2025.docx"
EXCEL_FILE = r"C:\Users\AGS_ASC\Desktop\coordinat\Processing\output.xlsx"

# Axtaracağımız başlıqlar (Word cədvəllərində "Növü", "Ərazi", və s.)
HEADINGS = [
    "Növü",
    "Ərazi",
    "İdarəçi şirkət",
    "Aqrotexniki və aqrokimyəvi vəziyyət",
    "Xəstəlik və zərərverici",
    "Suvarma",
    "Əkin sxemi",
    "Bağın yaşı",
    "Sahə (ha)"
]

# 1) Word sənədini oxuyuruq
doc = Document(WORD_FILE)

# 2) Excel WorkBook yaradıb, aktiv Worksheet götürürük
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Data"

# 3) Başlıq sətrini (Excel-in 1-ci sətirini) hazırlayaq
#    A1 hüceyrəsinə "ID" yazırıq, sonra HEADINGS siyahısındakı adlar
ws.cell(row=1, column=1, value="ID")
for i, heading in enumerate(HEADINGS, start=2):
    ws.cell(row=1, column=i, value=heading)

# Excel-ə məlumat yazacağımız növbəti sətir
excel_row = 2

# Hər cədvələ verəcəyimiz ID
table_id = 1

# 4) Word sənədindəki bütün cədvəlləri iterasiya edirik
for table in doc.tables:
    # Bu cədvəl üçün başlıqların qarşılığındakı (dəyərlər) toplayacağımız dictionary
    # Məsələn, {"Növü": "...", "Ərazi": "...", ...}
    table_data = {heading: "" for heading in HEADINGS}

    # Cədvəldəki hər sətri gəzirik
    for row in table.rows:
        # Hər sətirdəki bütün xanaların mətnini strip edirik
        cells = [cell.text.strip() for cell in row.cells]

        # Məsələn, bir sətir: [ "Növü", "Üzüm" ] və ya [ "Sahə (ha)", "0.8" ]
        # Biz 2 xanadan çox ola bilməsini də nəzərə alaraq yoxlamalar aparırıq:
        if len(cells) >= 2:
            # Birinci xanadakı mətni "başlıq" hesab edirik
            possible_heading = cells[0]
            # İkinci xanadakı mətni dəyər kimi götürürük
            value = cells[1]

            # Əgər bu "possible_heading" bizim HEADINGS siyahımızdakı bir başlıqdırsa,
            # dictionary-də saxlayırıq
            if possible_heading in table_data:
                table_data[possible_heading] = value

    # Cədvəldən bütün dəyərləri yığdıqdan sonra Excel-ə yazırıq
    #  - Birinci sütun (A) -> ID
    ws.cell(row=excel_row, column=1, value=table_id)

    #  - Qalan sütunlar -> HEADINGS siyahısındakı ardıcıllıqla dəyərlər
    for col_index, heading in enumerate(HEADINGS, start=2):
        ws.cell(row=excel_row, column=col_index, value=table_data[heading])

    # Sətir sayı və cədvəl ID-ni artırırıq
    excel_row += 1
    table_id += 1

# 5) Excel faylını saxlayırıq
wb.save(EXCEL_FILE)
print(f"Başlıqların qarşısındakı məlumatlar '{EXCEL_FILE}' faylına yazıldı.")


Başlıqların qarşısındakı məlumatlar 'C:\Users\AGS_ASC\Desktop\coordinat\Processing\output.xlsx' faylına yazıldı.
